In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


# ==========================================
# 1. DYNAMIC MULTI-SCALE BACKBONE (FIXED)
# ==========================================
class ResNetBackbone(nn.Module):
    """Dynamically extracts multi-scale P3, P4, P5 feature tracks from standard ResNets securely."""

    def __init__(self, name="resnet34", pretrained=True):
        super().__init__()
        base = getattr(models, name)(weights="DEFAULT" if pretrained else None)

        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1  # P2 (stride 4) - ignored for speed
        self.layer2 = base.layer2  # P3 (stride 8)
        self.layer3 = base.layer3  # P4 (stride 16)
        self.layer4 = base.layer4  # P5 (stride 32)

        # Dynamic channel tracking pass without corrupting module state
        self.out_channels = self._get_channels()

    def _get_channels(self):
        # Pass dummy forward cleanly through sequential steps
        was_training = self.training
        self.eval()  # Temporary switch
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            o1 = self.layer1(self.stem(dummy))
            o2 = self.layer2(o1)
            o3 = self.layer3(o2)
            o4 = self.layer4(o3)
            channels = [o2.shape[1], o3.shape[1], o4.shape[1]]

        if was_training:
            self.train()  # Revert back safely
        return channels

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        p3 = self.layer2(x)
        p4 = self.layer3(p3)
        p5 = self.layer4(p4)
        return [p3, p4, p5]


# ==========================================
# 2. FEATURE PYRAMID FUSION NECK (DYNAMIC SIZE FIX)
# ==========================================
class SimpleNeckFPN(nn.Module):
    """Standardizes incoming channels and propagates semantic maps top-down.

    Guards against odd-dimension tensor misalignments.
    """

    def __init__(self, in_channels, out_chan=256):
        super().__init__()
        self.p5_proj = nn.Conv2d(in_channels[2], out_chan, kernel_size=1)
        self.p4_proj = nn.Conv2d(in_channels[1], out_chan, kernel_size=1)
        self.p3_proj = nn.Conv2d(in_channels[0], out_chan, kernel_size=1)

    def forward(self, features):
        p3, p4, p5 = features

        f5 = self.p5_proj(p5)

        # Dynamic interpolation matches target shapes precisely, bypassing hard scale factors
        f4 = self.p4_proj(p4) + F.interpolate(f5, size=p4.shape[2:], mode="nearest")
        f3 = self.p3_proj(p3) + F.interpolate(f4, size=p3.shape[2:], mode="nearest")

        return f3, f4, f5


# ==========================================
# 3. UNIVERSAL DECOUPLED HEAD MASTER
# ==========================================
class UniversalDecoupledHead(nn.Module):
    """Highly flexible anchor-free head.

    Alters task configuration instantly based on target settings.
    """

    def __init__(self, num_classes, in_channels=256, reg_max=16, task="detection"):
        super().__init__()
        self.num_classes = num_classes
        self.reg_max = reg_max
        self.task = task

        if self.task == "detection":
            self.reg_dim = 4 * reg_max
        elif self.task == "rotated":
            self.reg_dim = (4 * reg_max) + 1
        elif self.task == "segmentation":
            self.reg_dim = 4 * reg_max
            self.proto_dim = 32

        self.cls_convs = nn.ModuleList(
            [self._make_block(in_channels) for _ in range(3)]
        )
        self.reg_convs = nn.ModuleList(
            [self._make_block(in_channels) for _ in range(3)]
        )

        self.cls_preds = nn.ModuleList(
            [nn.Conv2d(in_channels, num_classes, 1) for _ in range(3)]
        )
        self.reg_preds = nn.ModuleList(
            [nn.Conv2d(in_channels, self.reg_dim, 1) for _ in range(3)]
        )

        if self.task == "segmentation":
            self.seg_preds = nn.ModuleList(
                [nn.Conv2d(in_channels, self.proto_dim, 1) for _ in range(3)]
            )
            self.proto_head = nn.Sequential(
                nn.Conv2d(in_channels, in_channels, 3, padding=1),
                nn.SiLU(),
                nn.Conv2d(in_channels, self.proto_dim, 1),
            )

    def _make_block(self, channels):
        return nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.SiLU(),
        )

    def forward(self, pyramid_features):
        outputs = []
        proto = (
            self.proto_head(pyramid_features[0])
            if self.task == "segmentation"
            else None
        )

        for i in range(3):
            feat = pyramid_features[i]

            cls_feat = self.cls_convs[i](feat)
            reg_feat = self.reg_convs[i](feat)

            cls_out = self.cls_preds[i](cls_feat)
            reg_out = self.reg_preds[i](reg_feat)

            if self.task == "detection" or self.task == "rotated":
                outputs.append((cls_out, reg_out))
            elif self.task == "segmentation":
                seg_out = self.seg_preds[i](reg_feat)
                outputs.append((cls_out, reg_out, seg_out))

        return (outputs, proto) if self.task == "segmentation" else outputs


# ==========================================
# 4. WRAPPER PIPELINE
# ==========================================
class OmniDetector(nn.Module):
    def __init__(self, resnet_name="resnet34", num_classes=20, task="detection"):
        super().__init__()
        self.backbone = ResNetBackbone(name=resnet_name, pretrained=True)
        self.neck = SimpleNeckFPN(in_channels=self.backbone.out_channels, out_chan=256)
        self.head = UniversalDecoupledHead(num_classes=num_classes, task=task)

    def forward(self, x):
        return self.head(self.neck(self.backbone(x)))


In [8]:
model = OmniDetector(resnet_name="resnet34", num_classes=20, task="detection")
outputs = model(torch.randn(2, 3, 640, 640))
# Returns: List of 3 tuples -> [(cls_P3, reg_P3), (cls_P4, reg_P4), (cls_P5, reg_P5)]
# Use reg_out for Continuous Distribution Focal Loss (DFL) box parsing.

In [10]:
outputs[0]

(tensor([[[[ 2.3898e-01, -4.1657e-01, -2.2995e-02,  ...,  4.0363e-01,
            -3.2051e-02,  3.3187e-01],
           [ 6.2359e-01,  1.1613e-01,  3.1922e-01,  ...,  1.7950e-01,
            -3.2328e-02,  2.2036e-01],
           [ 4.6936e-01,  1.1023e-01,  4.7889e-01,  ...,  1.5556e-01,
             2.7714e-01,  6.3076e-01],
           ...,
           [-6.5980e-01, -4.4746e-01,  3.7035e-02,  ..., -7.4103e-04,
            -7.2387e-02,  1.6080e-01],
           [-4.0015e-01, -1.4672e-01,  7.2156e-02,  ...,  9.2817e-02,
             1.6172e-01,  2.6873e-01],
           [-1.3635e-02,  4.9132e-02,  3.6585e-02,  ..., -1.4335e-02,
            -2.9633e-02,  3.1905e-01]],
 
          [[ 3.5542e-01, -1.4764e-01, -5.1061e-01,  ..., -3.0399e-01,
            -2.7145e-01, -2.7656e-01],
           [ 1.0341e+00,  5.3407e-01,  4.8662e-01,  ..., -1.4698e-01,
            -3.2774e-01, -6.7529e-02],
           [ 6.0182e-01,  5.3812e-01,  8.6769e-01,  ...,  1.1743e-01,
            -9.0821e-02,  2.2572e-01],


In [ ]:
model = OmniDetector(resnet_name="resnet50", num_classes=15, task="rotated")
outputs = model(torch.randn(2, 3, 640, 640))
# Returns: List of 3 tuples -> [(cls_P3, reg_P3), ...]
# The regression tensor has 1 additional dimension containing the explicit unconstrained angle scalar logit.


In [ ]:
model = OmniDetector(resnet_name="resnet34", num_classes=80, task="segmentation")
outputs, proto = model(torch.randn(2, 3, 640, 640))
# Returns:
#   1. Outputs List -> [(cls_P3, reg_P3, coeff_P3), ...]
#   2. Proto Matrix -> [B, 32, H/8, W/8] (The global spatial mask prototypes)
# Action: Compute a matrix multiplication (Linear Combination) between the predicted
# cell coefficients (coeff) and the proto matrix to generate instance binary masks instantly.


In [ ]:
def decode_dfl_boxes(reg_out, reg_max=16):
    """Converts raw DFL tensor distribution configurations back into absolute bounding box boundaries."""
    B, _, H, W = reg_out.shape
    # Reshape channel allocation to separate boundaries and distribution slots
    reg_out = reg_out.reshape(B, 4, reg_max, H, W)
    prob = torch.softmax(reg_out, dim=2)

    # Compute expected distance value via scalar weights
    slots = torch.arange(reg_max, dtype=torch.float32, device=reg_out.device).view(
        1, 1, reg_max, 1, 1
    )
    ltrb_distances = torch.sum(prob * slots, dim=2)  # Yields Shape: [B, 4, H, W]
    return ltrb_distances
